In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Phase4") \
    .getOrCreate()

### Step 1: Read Datasets

In [0]:
customers = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("/Volumes/workspace/default/databricks2027/customers.csv")

sales = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("/Volumes/workspace/default/databricks2027/sales.csv")

display(customers)
display(sales)

customer_id,first_name,last_name,email,phone_number,address,city,state,zip_code
1,John,Smith,john.smith@domain.com,555-0001,123 Elm St,Springfield,IL,62701
2,Emma,Jones,emma.jones@webmail.com,555-0002,456 Oak St,Centerville,OH,45459
3,Olivia,Brown,olivia.brown@outlook.com,555-0003,789 Pine St,Greenville,SC,29601
4,Liam,Johnson,liam.johnson@gmail.com,555-0004,101 Maple St,Riverside,CA,92501
5,Noah,Williams,noah.williams@yahoo.com,555-0005,202 Birch St,Lakeside,TX,75001
6,Alice,Miller,alice.miller@aol.com,555-0006,303 Cedar St,Oakland,CA,94601
7,Isabella,Davis,isabella.davis@icloud.com,555-0007,404 Spruce St,Boise,ID,83701
8,James,Martinez,james.martinez@live.com,555-0008,505 Walnut St,Des Moines,IA,50301
9,Sophia,Garcia,sophia.garcia@zoho.com,555-0009,606 Cherry St,Albany,NY,12201
10,Lucas,Rodriguez,lucas.rodriguez@hotmail.com,555-0010,707 Maple St,Portland,OR,97201


sale_id,customer_id,product_id,sale_date,quantity,total_amount
1,1,1,2024-01-15,2,39.98
2,1,3,2024-01-20,1,29.99
3,2,2,2024-01-16,1,25.0
4,2,4,2024-01-22,3,89.97
5,3,5,2024-01-17,2,49.98
6,4,6,2024-01-18,4,119.96
7,4,7,2024-01-25,1,15.5
8,5,8,2024-01-19,3,66.75
9,6,9,2024-01-20,2,40.0
10,7,10,2024-01-21,5,110.95


### Step 2: Clean Data

In [0]:
customers_clean = customers.dropna().dropDuplicates()
sales_clean = sales.dropna().dropDuplicates()

### Step 3: Create SQL Views

In [0]:
customers_clean.createOrReplaceTempView("customers")
sales_clean.createOrReplaceTempView("sales")

### Task 1: Daily Sales

- SQL

In [0]:
%sql
SELECT
sale_date,
SUM(total_amount) AS daily_sales
FROM sales
GROUP BY sale_date
ORDER BY sale_date;

sale_date,daily_sales
2024-01-15,39.98
2024-01-16,25.0
2024-01-17,49.98
2024-01-18,119.96
2024-01-19,66.75
2024-01-20,69.99
2024-01-21,110.95
2024-01-22,109.97
2024-01-23,79.96
2024-01-24,55.0


- PySpark

In [0]:
from pyspark.sql.functions import sum

daily_sales = sales_clean.groupBy("sale_date") \
    .agg(sum("total_amount").alias("daily_sales")) \
    .orderBy("sale_date")

display(daily_sales)

sale_date,daily_sales
2024-01-15,39.98
2024-01-16,25.0
2024-01-17,49.98
2024-01-18,119.96
2024-01-19,66.75
2024-01-20,69.99
2024-01-21,110.95
2024-01-22,109.97
2024-01-23,79.96
2024-01-24,55.0


### Task 2: City-wise Revenue

- SQL

In [0]:
%sql
SELECT
c.city,
SUM(s.total_amount) AS city_revenue
FROM customers c
JOIN sales s
ON c.customer_id=s.customer_id
GROUP BY c.city
ORDER BY city_revenue DESC;

city,city_revenue
Riverside,135.45999999999998
Boston,119.96
Centerville,114.97
Las Vegas,112.47
Boise,110.95
New York,109.96
Seattle,104.99
St. Louis,99.95
San Diego,95.5
Washington,92.0


- PySpark

In [0]:
from pyspark.sql.functions import sum

customer_sales = customers_clean.join(
    sales_clean,
    "customer_id"
)

city_revenue = customer_sales.groupBy("city") \
    .agg(sum("total_amount").alias("city_revenue")) \
    .orderBy("city_revenue", ascending=False)

display(city_revenue)

city,city_revenue
Riverside,135.45999999999998
Boston,119.96
Centerville,114.97
Las Vegas,112.47
Boise,110.95
New York,109.96
Seattle,104.99
St. Louis,99.95
San Diego,95.5
Washington,92.0


### Task 3: Top 5 Customers by Revenue

- SQL

In [0]:
%sql
SELECT customer_id,SUM(total_amount) AS total_spend
FROM sales
GROUP BY customer_id
ORDER BY total_spend DESC
LIMIT 5;

customer_id,total_spend
4,135.45999999999998
22,119.96
2,114.97
7,110.95
45,109.96


- PySpark

In [0]:
top_customers = sales_clean.groupBy("customer_id") \
    .agg(sum("total_amount").alias("total_spend")) \
    .orderBy("total_spend", ascending=False)

display(top_customers.limit(5))

customer_id,total_spend
4,135.45999999999998
22,119.96
2,114.97
7,110.95
45,109.96


### Task 4: Repeat Customers

- SQL

In [0]:
%sql
SELECT customer_id,COUNT(sale_id) AS total_orders
FROM sales
GROUP BY customer_id
HAVING COUNT(sale_id)>2;

customer_id,total_orders


- PySpark

In [0]:
from pyspark.sql.functions import count

repeat_customers = sales_clean.groupBy("customer_id") \
    .agg(count("sale_id").alias("total_orders")) \
    .filter("total_orders > 2")

display(repeat_customers)

customer_id,total_orders


### Task 5: Customer Segmentation (Gold / Silver / Bronze)

- SQL

In [0]:
%sql
SELECT customer_id,SUM(total_amount) AS total_spend,
CASE
WHEN SUM(total_amount)>=5000 THEN 'Gold'
WHEN SUM(total_amount)>=2000 THEN 'Silver'
ELSE 'Bronze'
END AS customer_segment
FROM sales
GROUP BY customer_id;

customer_id,total_spend,customer_segment
4,135.45999999999998,Bronze
19,29.99,Bronze
22,119.96,Bronze
43,40.0,Bronze
1,69.97,Bronze
26,66.75,Bronze
2,114.97,Bronze
13,34.0,Bronze
18,59.97,Bronze
21,49.98,Bronze


- PySpark

In [0]:
from pyspark.sql.functions import when,col

customer_segment = sales_clean.groupBy("customer_id") \
    .agg(sum("total_amount").alias("total_spend")) \
    .withColumn(
        "customer_segment",
        when(col("total_spend") >= 5000, "Gold")
        .when(col("total_spend") >= 2000, "Silver")
        .otherwise("Bronze")
    )

display(customer_segment)

customer_id,total_spend,customer_segment
4,135.45999999999998,Bronze
19,29.99,Bronze
22,119.96,Bronze
43,40.0,Bronze
1,69.97,Bronze
26,66.75,Bronze
2,114.97,Bronze
13,34.0,Bronze
18,59.97,Bronze
21,49.98,Bronze


### Task 6: Final Reporting Table

- SQL 

In [0]:
%sql
SELECT c.customer_id,c.first_name,c.last_name,c.city,COUNT(s.sale_id) AS total_orders,SUM(s.total_amount) AS total_spend
FROM customers c
JOIN sales s
ON c.customer_id=s.customer_id
GROUP BY c.customer_id,c.first_name,c.last_name,c.city
ORDER BY total_spend DESC;

customer_id,first_name,last_name,city,total_orders,total_spend
4,Liam,Johnson,Riverside,2,135.45999999999998
22,Henry,Moore,Boston,1,119.96
2,Emma,Jones,Centerville,2,114.97
7,Isabella,Davis,Boise,1,110.95
45,Charlotte,Wood,New York,1,109.96
24,Daniel,Walker,St. Louis,1,99.95
15,Harper,Jackson,Seattle,1,92.0
34,Lucas,Mitchell,Washington,1,92.0
20,Elijah,Garcia,Detroit,1,89.97
39,Aria,Davis,Memphis,1,89.96


In [0]:
from pyspark.sql.functions import sum, count

final_report = customer_sales.groupBy(
    "customer_id",
    "first_name",
    "last_name",
    "city"
).agg(
    count("sale_id").alias("total_orders"),
    sum("total_amount").alias("total_spend")
).orderBy("total_spend", ascending=False)

display(final_report)

customer_id,first_name,last_name,city,total_orders,total_spend
4,Liam,Johnson,Riverside,2,135.45999999999998
22,Henry,Moore,Boston,1,119.96
2,Emma,Jones,Centerville,2,114.97
7,Isabella,Davis,Boise,1,110.95
45,Charlotte,Wood,New York,1,109.96
24,Daniel,Walker,St. Louis,1,99.95
15,Harper,Jackson,Seattle,1,92.0
34,Lucas,Mitchell,Washington,1,92.0
20,Elijah,Garcia,Detroit,1,89.97
39,Aria,Davis,Memphis,1,89.96


### Task 7: Save Final Report

In [0]:
final_report.write.mode("overwrite") \
    .option("header", True) \
    .csv("/Volumes/workspace/default/databricks2027/final_report")

# Reflection Questions

## 1. Why is cleaning done before joining tables?

Cleaning is performed before joining tables to remove null values, duplicate records, and invalid data. This ensures accurate joins, prevents duplicate results, and improves the overall quality and reliability of the final output.

---

## 2. What would go wrong if null keys are not removed?

If null values exist in the join key (such as `customer_id`), the records may not match correctly during the join operation. This can lead to missing records, incorrect results, and inaccurate business reports.

---

## 3. How did you decide the join order?

The `customers` dataset was used as the master dataset because it contains customer information, while the `sales` dataset contains transaction details. Both datasets were joined using the common key `customer_id` to combine customer details with their sales information.

---

## 4. Which step was most difficult and why?

The customer segmentation and aggregation steps were the most challenging because they required grouping the data, calculating total spending, and applying business rules to classify customers into different categories.

---

## 5. How is SQL logic similar to PySpark?

SQL and PySpark follow the same logical operations. SQL statements such as `SELECT`, `WHERE`, `GROUP BY`, `JOIN`, and `ORDER BY` have equivalent PySpark DataFrame operations like `select()`, `filter()`, `groupBy()`, `join()`, and `orderBy()`. Both approaches produce the same results using different syntax.

---

## 6. What challenges will appear with large data?

When processing large datasets, challenges may include:

- Data skew
- Memory limitations
- Long execution time
- Expensive shuffle operations
- Slow joins
- Increased storage requirements

These challenges can be addressed using Spark optimizations such as partitioning, caching, broadcast joins, and efficient resource management.

---

## 7. Can you explain your pipeline in simple steps?

The ETL pipeline can be explained in the following steps:

1. **Extract** – Read customer and sales datasets from CSV files.
2. **Transform** – Clean the data by removing null values and duplicates, then join the datasets and perform business transformations such as aggregations and customer segmentation.
3. **Load** – Display the final reports and save the processed data for further analysis or reporting.

This workflow demonstrates a complete ETL pipeline using PySpark, following industry-standard data engineering practices.